# Billing System (prefix sums + a clamped rate)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Arrays, Math · **Difficulty/Frequency:** Rare (2/10)

> **Language note.** The official answer is Java; this notebook implements the same design in Python so every claim is executable. The Java reference is preserved verbatim in [`README.md`](README.md).

## Concepts

**What this problem is really testing:**
- **Prefix sums** — turning "sum a range" from O(n) into O(1)
- Reading a **specification** precisely, and noticing where it is ambiguous or commercially wrong
- The **update-vs-query trade-off**, and knowing the structure that resolves it

**First-principles primer — what is each piece?**

- **Prefix sum.** `prefix[i] = bill[0] + ... + bill[i-1]`. Build it once in O(n), and thereafter the sum of *any* range is a single subtraction:

  ```
  sum(bill[l..r]) = prefix[r+1] - prefix[l]
  ```

  The `+1` offset — `prefix[0] = 0` for the empty range — is what makes `l = 0` work without a special case, exactly like the half-open interval in binary search.

- **`max(minimum, rate × usage)`.** A **piecewise-linear** cost: flat while usage is small, proportional once it crosses the break-even point `minimum / rate`. Every phone tariff and cloud bill has this shape. It is the "you pay at least this much" clause written as arithmetic.

- **Plans partition the month.** Days 0–10, 11–15, 16–30. Each day is billed under exactly one plan, so the plans' usages add up to the total usage with no double-counting — which is *why* summing per-plan contributions is legitimate.

**The core move:** for a query at day `d`, plan `i` only sees the days in the **intersection** of its range with `[0, d]`:

```
effective_start = max(0, plan.start)
effective_end   = min(d, plan.end)
usage_i         = prefix[effective_end + 1] - prefix[effective_start]
```

That clamping is the entire multi-plan logic. Everything else is the same formula applied per plan.

**The specification problem worth raising.** The official answer notes it and then codes past it:

> *"`day` before any plan starts: plan cost is `max(min, 0) = minimumPlanCost`. The customer pays the minimum even with zero usage."*

Query on **day 3** in the three-plan example and you charge `v1 + v2 + v3` — billing for two plans that have not started. No real billing system does that. It is not a rounding detail; it is a different bill. Ask, and implement the answer.

**Simple worked example.** `bill = [10]*31`, plans as in the prompt with `min = 5`, `rate = 0.1`, query day **17**:

| plan | days used | usage | `rate × usage` | plan cost = `max(5, ·)` |
|---|---|---|---|---|
| 0–10 | 0–10 (11 days) | 110 | 11.0 | **11.0** |
| 11–15 | 11–15 (5 days) | 50 | 5.0 | **5.0** ← exactly at break-even |
| 16–30 | 16–**17** (clamped!) | 20 | 2.0 | **5.0** ← the minimum bites |

Total usage `110 + 50 + 20 = 180`; plan costs `11 + 5 + 5 = 21`; **total = 201**.

Note the third plan: its range runs to day 30, but the query clamps it at 17 — and its small usage falls below the **break-even point** (`minimum / rate = 5.0 / 0.1 = 50` units), so the *minimum* applies rather than the rate. The second plan lands exactly *on* that point, where both branches of the `max` give the same answer — which is precisely the case worth testing, because it is where an off-by-one in the comparison would hide.

## Problem Statement

**Problem 1** — `update(day, cost)`: `bill[day] += cost`.

**Problem 2** — the total up to and including a day, for a single plan:

```
usage      = sum(bill[0..day])
plan_cost  = max(MinimumPlanCost, usage * percentage)
total      = usage + plan_cost
```

**Problem 3** — several plans, each `{start, end, MinimumPlanCost, percentage}`, partitioning the month. For a query day, apply the formula to each plan over the part of its range that has elapsed, and sum.

### Approach 1 — Naive (re-sum the array on every query)

**Idea:** loop from day 0 to the query day, adding as you go.

Perfectly fine for a single query, and the right thing to write first. It becomes the bottleneck the moment there are many queries or many plans — every plan re-walks its own slice of the array, every time.

**Time complexity:** O(1) per update; **O(n)** per query (O(n × p) for p plans).

**Space complexity:** O(1) beyond the bill.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional


@dataclass
class Plan:
    start: int
    end: int
    minimum_plan_cost: float
    percentage: float


class NaiveBilling:
    """Baseline: sums the array on every query."""

    def __init__(self, days: int = 31, plans: Optional[List[Plan]] = None) -> None:
        self.bill = [0] * days
        self.plans = plans or []

    def update(self, day: int, cost: float) -> None:
        self.bill[day] += cost

    def total_cost(self, day: int) -> float:
        total = 0.0
        for p in self.plans:
            lo, hi = max(0, p.start), min(day, p.end)
            usage = sum(self.bill[lo:hi + 1]) if lo <= hi else 0.0   # O(n) EVERY query
            total += usage + max(p.minimum_plan_cost, p.percentage * usage)
        return total

### Approach 2 — Optimal (prefix sums)

**Idea:** build `prefix` once; every range sum is then one subtraction.

Two details worth defending:

- **`prefix` has `n + 1` entries, with `prefix[0] = 0`.** That extra leading zero is what makes `rangeSum(0, r)` work without a special case — the same trick as the half-open interval in binary search. Sizing it `n` instead forces an `if l == 0` branch that people then get wrong.
- **`if lo > hi: return 0`.** When a plan has not started (or the query precedes its range) the intersection is **empty**, and an empty range must contribute zero usage — not a negative number from a reversed subtraction.

**The policy flag.** `charge_unstarted_plans` makes the ambiguity explicit rather than silently picking one. With it `False` (the default here), a plan contributes only once `day >= plan.start` — which is what a billing system actually wants. With it `True` you reproduce the official answer exactly.

**Time complexity:** **O(1)** per range sum, so **O(p)** per query; O(n) to build the prefix.

**Space complexity:** O(n).

In [ ]:
class BillingSystem:
    """Prefix sums for O(1) range queries, with an explicit unstarted-plan policy."""

    def __init__(self, days: int = 31, plans: Optional[List[Plan]] = None,
                 charge_unstarted_plans: bool = False) -> None:
        self.bill = [0.0] * days
        self.plans = plans or []
        self.charge_unstarted_plans = charge_unstarted_plans   # make the AMBIGUITY explicit
        self._prefix: Optional[List[float]] = None             # invalidated on update

    # ---- Problem 1 ----
    def update(self, day: int, cost: float) -> None:
        if not 0 <= day < len(self.bill):
            raise IndexError(f"day {day} outside 0..{len(self.bill) - 1}")
        self.bill[day] += cost
        self._prefix = None                    # lazy: rebuild only when a query needs it

    def _build_prefix(self) -> List[float]:
        if self._prefix is None:
            # n+1 entries with a leading 0, so rangeSum(0, r) needs no special case.
            pref = [0.0] * (len(self.bill) + 1)
            for i, v in enumerate(self.bill):
                pref[i + 1] = pref[i] + v
            self._prefix = pref
        return self._prefix

    def range_sum(self, lo: int, hi: int) -> float:
        """Inclusive sum of bill[lo..hi]; 0.0 if the range is empty."""
        if lo > hi:
            return 0.0                         # an EMPTY intersection contributes nothing
        pref = self._build_prefix()
        lo, hi = max(0, lo), min(len(self.bill) - 1, hi)
        return pref[hi + 1] - pref[lo]

    # ---- Problem 2 ----
    def usage_cost(self, day: int) -> float:
        return self.range_sum(0, day)

    @staticmethod
    def plan_cost(usage: float, minimum: float, percentage: float) -> float:
        return max(minimum, usage * percentage)          # piecewise-linear: flat, then proportional

    def single_plan_total(self, day: int, minimum: float, percentage: float) -> float:
        usage = self.usage_cost(day)
        return usage + self.plan_cost(usage, minimum, percentage)

    # ---- Problem 3 ----
    def total_cost(self, day: int) -> float:
        total = 0.0
        for p in self.plans:
            if not self.charge_unstarted_plans and day < p.start:
                continue                        # the plan has not begun - no minimum charge
            lo, hi = max(0, p.start), min(day, p.end)     # the INTERSECTION with [0, day]
            usage = self.range_sum(lo, hi)
            total += usage + self.plan_cost(usage, p.minimum_plan_cost, p.percentage)
        return total

    def breakdown(self, day: int) -> List[dict]:
        """Per-plan detail - what an invoice would actually show."""
        rows = []
        for p in self.plans:
            if not self.charge_unstarted_plans and day < p.start:
                continue
            lo, hi = max(0, p.start), min(day, p.end)
            usage = self.range_sum(lo, hi)
            cost = self.plan_cost(usage, p.minimum_plan_cost, p.percentage)
            rows.append({"start": p.start, "end": p.end, "days": max(0, hi - lo + 1),
                         "usage": usage, "plan_cost": cost,
                         "at_minimum": cost == p.minimum_plan_cost, "total": usage + cost})
        return rows

### Approach 3 — A Fenwick tree, for a write-heavy workload

**Idea:** prefix sums make queries O(1) but every update invalidates the whole array — O(n) to rebuild. That is the wrong trade when updates outnumber queries, which for a *live* billing system they do.

A **Fenwick tree** (binary indexed tree) gets **O(log n) for both**. The idea: each slot stores the sum of a block whose length is the lowest set bit of its index, so a prefix sum is assembled from at most `log n` blocks, and an update touches at most `log n` of them.

For a 31-day month this is academic — `log 31` ≈ 5 versus `n` = 31, and the constant factors probably favour the array. It becomes the right answer when the range is a year of minutes, or millions of accounts. **Knowing when *not* to reach for it is as much of the answer as knowing how.**

**Time complexity:** **O(log n)** per update and per prefix query.

**Space complexity:** O(n).

In [ ]:
class FenwickBilling(BillingSystem):
    """Same API, O(log n) updates - worth it only when writes dominate."""

    def __init__(self, days: int = 31, plans: Optional[List[Plan]] = None,
                 charge_unstarted_plans: bool = False) -> None:
        super().__init__(days, plans, charge_unstarted_plans)
        self._tree = [0.0] * (days + 1)        # 1-indexed internally

    def update(self, day: int, cost: float) -> None:
        if not 0 <= day < len(self.bill):
            raise IndexError(f"day {day} outside 0..{len(self.bill) - 1}")
        self.bill[day] += cost
        i = day + 1
        while i < len(self._tree):
            self._tree[i] += cost
            i += i & (-i)                      # jump to the next block that covers this index

    def _prefix_sum(self, day: int) -> float:
        """Sum of bill[0..day]."""
        total, i = 0.0, day + 1
        while i > 0:
            total += self._tree[i]
            i -= i & (-i)                      # strip the lowest set bit: at most log n steps
        return total

    def range_sum(self, lo: int, hi: int) -> float:
        if lo > hi:
            return 0.0
        lo, hi = max(0, lo), min(len(self.bill) - 1, hi)
        return self._prefix_sum(hi) - (self._prefix_sum(lo - 1) if lo > 0 else 0.0)

## Verification

The worked example computed by hand, then the cases that matter: the minimum clamping in, the range clamping at the query day, unstarted plans under both policies, and agreement between all three implementations.

In [ ]:
import random

PLANS = [Plan(0, 10, 5.0, 0.1), Plan(11, 15, 5.0, 0.1), Plan(16, 30, 5.0, 0.1)]

# --- Problem 1: update accumulates, it does not overwrite ---
b = BillingSystem()
b.update(0, 10)
b.update(0, 5)
assert b.bill[0] == 15, "update ADDS to the day's cost"
try:
    b.update(31, 1)
except IndexError:
    pass
else:
    raise AssertionError("a day outside the month must be rejected")

# --- Problem 2: the single-plan formula ---
s = BillingSystem()
for d in range(31):
    s.update(d, 10)
assert s.usage_cost(0) == 10
assert s.usage_cost(4) == 50, "days 0..4 inclusive"
assert s.usage_cost(30) == 310

# rate * usage is above the minimum -> the rate applies
assert s.single_plan_total(4, minimum=1.0, percentage=0.1) == 50 + 5.0
# rate * usage is BELOW the minimum -> the minimum applies
assert s.single_plan_total(4, minimum=20.0, percentage=0.1) == 50 + 20.0
assert BillingSystem.plan_cost(0, 5.0, 0.1) == 5.0, "zero usage still pays the minimum"
assert BillingSystem.plan_cost(100, 5.0, 0.1) == 10.0, "above break-even, the rate wins"
# The break-even point is exactly minimum / rate
assert BillingSystem.plan_cost(50, 5.0, 0.1) == 5.0, "exactly at break-even: both give 5.0"

# --- Problem 3: the worked example, computed by hand ---
m = BillingSystem(plans=PLANS)
for d in range(31):
    m.update(d, 10)

rows = m.breakdown(17)
assert [r["days"] for r in rows] == [11, 5, 2], "the third plan is CLAMPED at day 17"
assert [r["usage"] for r in rows] == [110, 50, 20]
assert [r["plan_cost"] for r in rows] == [11.0, 5.0, 5.0]
assert [r["at_minimum"] for r in rows] == [False, True, True], (
    "plan 2 sits EXACTLY at break-even (50 * 0.1 == 5.0 == minimum); plan 3 is below it"
)
assert rows[0]["plan_cost"] > PLANS[0].minimum_plan_cost, "plan 1 is above break-even"
assert rows[1]["usage"] * PLANS[1].percentage == PLANS[1].minimum_plan_cost, (
    "the break-even point is exactly minimum / rate = 5.0 / 0.1 = 50 units of usage"
)
assert rows[2]["usage"] * PLANS[2].percentage < PLANS[2].minimum_plan_cost, "plan 3 is below"
assert m.total_cost(17) == 180 + 21.0, m.total_cost(17)

# Querying the last day covers every plan fully
assert sum(r["days"] for r in m.breakdown(30)) == 31
assert m.total_cost(30) == 310 + (11.0 + 5.0 + 15.0)

# --- THE ambiguity: plans that have not started yet ---
early = BillingSystem(plans=PLANS, charge_unstarted_plans=False)
official = BillingSystem(plans=PLANS, charge_unstarted_plans=True)
for d in range(31):
    early.update(d, 10)
    official.update(d, 10)

assert len(early.breakdown(3)) == 1, "only the plan covering day 3 is billed"
assert len(official.breakdown(3)) == 3, "the official version bills all three"
assert early.total_cost(3) == 40 + 5.0, "usage 40, one plan at the minimum"
assert official.total_cost(3) == 40 + 5.0 + 5.0 + 5.0, (
    "the official reading charges two plans that have not begun"
)
assert official.total_cost(3) > early.total_cost(3), "the policy genuinely changes the bill"
# Once every plan has started, the two policies agree
assert early.total_cost(30) == official.total_cost(30)
assert early.total_cost(16) == official.total_cost(16)

# --- Uneven daily costs, and a zero-usage month ---
uneven = BillingSystem(plans=PLANS)
uneven.update(0, 100)
uneven.update(20, 50)
assert uneven.range_sum(0, 10) == 100
assert uneven.range_sum(11, 15) == 0
assert uneven.range_sum(16, 30) == 50
assert uneven.total_cost(30) == 150 + (10.0 + 5.0 + 5.0), uneven.total_cost(30)

empty = BillingSystem(plans=PLANS)
assert empty.total_cost(30) == 0 + 15.0, "no usage at all: three minimums"
assert BillingSystem(plans=[]).total_cost(30) == 0.0, "no plans at all"

# --- Range clamping, including empty intersections ---
r = BillingSystem(plans=PLANS)
for d in range(31):
    r.update(d, 1)
assert r.range_sum(5, 3) == 0.0, "a reversed range is empty, not negative"
assert r.range_sum(0, 0) == 1.0
assert r.range_sum(0, 30) == 31.0
assert r.range_sum(-5, 3) == 4.0, "a low bound below 0 is clamped"
assert r.range_sum(28, 99) == 3.0, "a high bound past the month is clamped"

# --- Negative adjustments (a refund) ---
refund = BillingSystem(plans=[Plan(0, 30, 0.0, 0.1)])
refund.update(0, 100)
refund.update(1, -30)
assert refund.usage_cost(30) == 70, "a credit reduces the usage"
assert refund.total_cost(30) == 70 + 7.0

# --- All three implementations agree, on randomised months ---
random.seed(89)
for _ in range(300):
    plans = [Plan(0, 10, random.uniform(0, 20), random.uniform(0, 0.3)),
             Plan(11, 15, random.uniform(0, 20), random.uniform(0, 0.3)),
             Plan(16, 30, random.uniform(0, 20), random.uniform(0, 0.3))]
    charges = [(random.randrange(31), random.uniform(-20, 100)) for _ in range(30)]

    fast = BillingSystem(plans=plans, charge_unstarted_plans=True)
    naive = NaiveBilling(plans=plans)
    fen = FenwickBilling(plans=plans, charge_unstarted_plans=True)
    for day, cost in charges:
        fast.update(day, cost)
        naive.update(day, cost)
        fen.update(day, cost)

    for q in range(31):
        a, b_, c = fast.total_cost(q), naive.total_cost(q), fen.total_cost(q)
        assert abs(a - b_) < 1e-9, (q, a, b_)
        assert abs(a - c) < 1e-9, (q, a, c)
        assert abs(fast.usage_cost(q) - fen.usage_cost(q)) < 1e-9

# --- The prefix cache must be invalidated by an update ---
inv = BillingSystem(plans=[Plan(0, 30, 0.0, 0.1)])
inv.update(0, 10)
assert inv.usage_cost(30) == 10          # builds the prefix
inv.update(5, 25)                        # must invalidate it
assert inv.usage_cost(30) == 35, "a stale prefix cache would still report 10"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Overlapping plans.** The range-sum machinery copes, but the *arithmetic* stops being valid: if day 12 is covered by two plans, its usage is counted twice in `sum(usage_i)`, and the "total usage" line of the invoice no longer matches the actual charges. You need an **allocation rule** first — prorate the day's usage between the plans, or bill it to whichever plan is more specific — and only then can the per-plan sum be meaningful. This is a **specification** gap, not an implementation one, which is exactly why it belongs in the "clarify before coding" list.
- **Update-heavy workloads.** Implemented above as `FenwickBilling`. The honest framing is a table: prefix sums are O(1) query / O(n) update; a Fenwick tree is O(log n) / O(log n); a plain array is O(n) / O(1). Pick by the read/write ratio. **For a 31-element array none of it matters** — `log 31 ≈ 5` versus `n = 31` — and saying so is better than reflexively reaching for the fancier structure. It earns its place when the range is a year of minutes or millions of accounts.
- **A variable month length.** Already parameterised (`days=31`). The subtlety a real system hits is not the array size but **February**: a plan defined as "days 0–30" is meaningless in a 28-day month, so plan ranges have to be validated and clamped against the actual month — otherwise `min(day, plan.end)` silently reads past the end.
- **Arbitrary date ranges, not just month-to-date.** `range_sum(l, r)` already supports it. What does not carry over is the **plan cost**: `MinimumPlanCost` is a *monthly* minimum, so charging it for an arbitrary window is wrong. You would need a proration rule — minimum × (days in window / days in plan) is the usual one — and that is again a business decision rather than a coding one.
- **Persistence.** A `daily_charges(account_id, day, amount)` table with `SUM(...) WHERE day <= ?` does the job, and the database's own index gives you the range scan. The prefix array is then a **cache**, with the same invalidation problem as the `_prefix = None` line above — just distributed, and therefore harder. For a month of 31 rows, a plain `SUM` is almost certainly faster than maintaining a materialised prefix.
- **Floating-point money.** The one thing this whole design gets wrong for production. `0.1 + 0.2 != 0.3` in binary floating point, and a billing system that is a hundredth of a penny out will eventually be a hundredth of a penny out *in someone's favour*, repeatedly. Real systems store **integer minor units** (pence, cents) or use `decimal.Decimal`, and define rounding explicitly at the point of charge. The tests above use `1e-9` tolerances precisely because floats were used — which is itself the argument.

## Empirical complexity check

Compare **re-summing the array per query** with **prefix sums**, over a growing number of days and a fixed query load.

| Growth when the month length doubles | What it means |
|---|---|
| ~2x | linear — every query re-walks the array |
| ~1x | constant per query — one subtraction per plan |

The Fenwick tree is included to make the trade visible from the other side: it is **slower** than plain prefix sums for queries (O(log n) rather than O(1)), and would only win on a workload dominated by updates.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random

QUERIES = 500


def make_month(days):
    """A `days`-long month split into 3 plans, with a charge on every day."""
    third = days // 3
    plans = [Plan(0, third, 5.0, 0.1),
             Plan(third + 1, 2 * third, 5.0, 0.1),
             Plan(2 * third + 1, days - 1, 5.0, 0.1)]
    rng = random.Random(97)
    charges = [(d, rng.uniform(1, 100)) for d in range(days)]
    queries = [rng.randrange(days) for _ in range(QUERIES)]
    return (days, plans, charges, queries)


def _run(cls, days, plans, charges, queries):
    b = cls(days=days, plans=plans) if cls is not NaiveBilling else cls(days=days, plans=plans)
    for d, c in charges:
        b.update(d, c)
    for q in queries:
        b.total_cost(q)


def run_naive(days, plans, charges, queries):
    _run(NaiveBilling, days, plans, charges, queries)      # O(n) per query


def run_prefix(days, plans, charges, queries):
    _run(BillingSystem, days, plans, charges, queries)     # O(1) per range sum


def run_fenwick(days, plans, charges, queries):
    _run(FenwickBilling, days, plans, charges, queries)    # O(log n) per range sum


benchmark(
    {"Approach 1 - re-sum per query O(n)": run_naive,
     "Approach 2 - prefix sums O(1)": run_prefix,
     "Approach 3 - Fenwick tree O(log n)": run_fenwick},
    make_month,
    sizes=[300, 600, 1200, 2400],
    repeats=2,
)

## Patterns learned

- **Precompute once, answer many times.** A prefix array turns every range sum into one subtraction. The same shape as the hash index in [Inverted Index](../1.%20Inverted_Index/1.%20Inverted_Index.ipynb) — pay O(n) at write time so reads are free.
- **Give your prefix array a leading zero.** `prefix[0] = 0` and `n+1` entries makes `rangeSum(0, r)` need no special case, exactly like the half-open interval in binary search.
- **Clamp the intersection, and let an empty range mean zero.** `max(0, start)`, `min(day, end)`, and `if lo > hi: return 0`. Without that last line a reversed range produces a *negative* sum, which in a billing system means a refund nobody authorised.
- **`max(minimum, rate × usage)` is a piecewise-linear tariff.** Flat below the break-even point `minimum / rate`, proportional above it. Recognising the shape tells you which test cases matter: below, at, and above the knee.
- **Match the structure to the read/write ratio.** Array: O(n) query, O(1) update. Prefix sums: O(1) query, O(n) update. Fenwick: O(log n) both. And for n = 31, use the array — knowing when *not* to reach for the clever structure is part of the answer.
- **A cache needs invalidating.** `self._prefix = None` on every update. One forgotten line and every query afterwards silently reports stale totals.
- **Read the spec for what it charges, not just what it computes.** The "unstarted plans still pay their minimum" reading is arithmetically faithful and commercially absurd. Ambiguity in a *money* spec is the thing to raise first, not last.
- **Never bill in floating point.** `0.1 + 0.2 != 0.3`. Use integer minor units or `Decimal`, and define rounding explicitly.